
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Detección de discurso de odio con Transformers: pysentimiento

# Introducción

En la notebook de [Vectorización](<../cap0/00 - SICSS-BAires - Vectorización.ipynb>) (cap0) construimos features a mano (TF, TF-IDF, embeddings promediados) y entrenamos una regresión logística LASSO para detectar discurso de odio en tweets del dataset **HatEval** (SemEval-2019 Task 5). En esta notebook resolvemos la **misma tarea, sobre el mismo split de test**, pero con un enfoque completamente distinto: en vez de diseñar features, usamos [`pysentimiento`](https://github.com/pysentimiento/pysentimiento), una librería que envuelve modelos **Transformer** (basados en RoBERTuito, un BERT preentrenado sobre tuits en español) ya *fine-tuneados* para tareas de análisis de texto en español y portugués — entre ellas, detección de discurso de odio.

La pregunta que guía esta notebook es: **¿cuánto ganamos al pasar de features hechas a mano a representaciones contextuales aprendidas por un Transformer, sobre exactamente los mismos datos?**

## Los datos

Usamos `data/hateval_test_df.csv`, el split de **test** de HatEval — el mismo que cap0 usó únicamente al final, para evaluar. Cada fila es un tweet con las columnas relevantes:

- `id`: identificador del tweet.
- `text`: el texto del tweet.
- `language`: idioma (`es` o `en`). Acá trabajamos solo con `es`.
- `HS`: discurso de odio (*Hate Speech*), 1 si el tweet es odioso, 0 si no.
- `TR`: *Target Range* — 1 si el odio está dirigido a un individuo específico, 0 si es genérico o grupal. Solo tiene sentido cuando `HS=1`.
- `AG`: *Aggressiveness* — 1 si el tweet es agresivo, 0 si no. También solo tiene sentido cuando `HS=1`.
- `target`: el grupo objetivo del tweet, `mig` (migrantes) o `mis` (mujeres, misoginia).

## Un caveat importante sobre esta evaluación

El modelo de `hate_speech` en español de pysentimiento (`robertuito-hate-speech`) fue **fine-tuneado sobre el split de train de este mismo dataset HatEval** — el mismo `hateval_train_df.csv` que usamos en cap0. Es decir: evaluar acá sobre `hateval_test_df.csv` es una evaluación **in-domain sobre el test oficial held-out**, no una prueba de generalización a un corpus distinto (como los tweets de campaña en `data/tweets_*.zip`). Esto hace que la comparación contra cap0 sea honesta (mismo train, mismo test, mismo idioma), pero también significa que estos números **no** nos dicen qué tan bien funcionaría el modelo sobre, por ejemplo, comentarios de un chat de WhatsApp o tuits de otro país o época.

## Qué vamos a hacer

1. Cargar el split de test y quedarnos con los tweets en español.
2. Entender cómo pysentimiento representa sus predicciones (`AnalyzerOutput`).
3. Predecir discurso de odio sobre los ~1.600 tweets en español del test.
4. Evaluar el desempeño con las mismas métricas que usamos en cap0.
5. Mirar en qué casos se equivoca el modelo y cómo varía su desempeño según el `target`.
6. Comparar contra los resultados de TF / TF-IDF / embeddings + LASSO de cap0.

In [ ]:
## Ejecutar para instalar pysentimiento y clonar el repo con los datos
## (en Colab: activá GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU)
!pip install pysentimiento
!git clone https://github.com/gefero/factor_data_tuto_NLP_SICSS.git

In [ ]:
# Importamos las librerías necesarias
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve,
)
from pysentimiento import create_analyzer
from pysentimiento.preprocessing import preprocess_tweet
import warnings
warnings.filterwarnings('ignore')

# Carga de los datos

In [ ]:
# Soporta correr tanto desde el repo recién clonado (Colab) como localmente
test_path = './factor_data_tuto_NLP_SICSS/data/hateval_test_df.csv'
if not os.path.exists(test_path):
    test_path = '../data/hateval_test_df.csv'

# Cargamos el split de test y nos quedamos con los tweets en español,
# igual que en cap0
def load_hateval_split(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    return df

test = load_hateval_split(test_path)
print(f'Tweets en español en el test: {len(test)}')
test[['id', 'text', 'target', 'HS', 'TR', 'AG']].head()

## Preprocesamiento

A diferencia de cap0, acá **no** aplicamos un preprocesamiento agresivo (sacar acentos, reemplazar números por `DIGITO`, pasar todo a minúsculas): ese tipo de limpieza está pensado para reducir el vocabulario de un `CountVectorizer`/`TfidfVectorizer`, pero un Transformer preentrenado sobre tuits reales aprovecha justamente esas señales (mayúsculas, acentos, emojis) que ese pipeline destruye.

En su lugar usamos `preprocess_tweet` de pysentimiento, que normaliza el texto de la misma forma en que se preprocesaron los tuits con los que se entrenó el modelo: reemplaza menciones por `@usuario`, URLs por `url`, separa hashtags y convierte emojis a su descripción textual.

In [ ]:
test['text_prep'] = test['text'].apply(lambda t: preprocess_tweet(t, lang='es'))

# Comparamos texto original vs. preprocesado en algunos ejemplos
test[['text', 'text_prep']].sample(3, random_state=42)

In [ ]:
# Distribución de las etiquetas en el test
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
fig.patch.set_facecolor('#fcfcfb')

for ax, col, title in zip(axes, ['HS', 'target'], ['Discurso de odio (HS)', 'Target']):
    counts = test[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=['#2a78d6', '#e34948'][:len(counts)])
    ax.set_facecolor('#fcfcfb')
    ax.set_title(title)
    ax.tick_params(colors='#898781')
    for spine in ax.spines.values():
        spine.set_color('#898781')

plt.tight_layout()
plt.show()

# Cómo funciona el analyzer

Antes de correr las predicciones sobre todo el test, veamos qué produce `pysentimiento` sobre unos pocos tweets de ejemplo (no forman parte del dataset, son solo para ilustrar).

In [ ]:
# Creamos el analizador de discurso de odio en español
# La primera vez descarga el modelo preentrenado (robertuito-hate-speech) desde HuggingFace
analyzer = create_analyzer(task="hate_speech", lang="es")

ejemplo_tweets = [
    "Qué lindo día para salir a caminar por Buenos Aires",
    "Estos inmigrantes vienen a robarnos el trabajo, hay que echarlos a todos",
    "No puedo creer que perdimos el partido, qué bronca",
    "Las mujeres no deberían poder votar, son todas unas histéricas",
]

resultados_ejemplo = analyzer.predict(ejemplo_tweets)
for tweet, res in zip(ejemplo_tweets, resultados_ejemplo):
    print(f'Tweet: {tweet}')
    print(f'  output (etiquetas activas): {res.output}')
    print(f'  probas: {res.probas}')
    print()

La tarea de `hate_speech` es **multilabel**: `res.output` es una lista con cero, una o más etiquetas entre `hateful`, `targeted` y `aggressive`, y `res.probas` es un diccionario con la probabilidad de cada una. El mapeo con las columnas de HatEval es directo:

| pysentimiento | HatEval |
|---|---|
| `hateful` | `HS` |
| `targeted` | `TR` |
| `aggressive` | `AG` |

En esta notebook nos enfocamos en `hateful` vs. `HS` (la tarea binaria principal); en el ejercicio final se propone evaluar también `targeted` y `aggressive`.

# Predicción sobre el test

Corremos el analyzer sobre los ~1.600 tweets en español del test, en batch (mucho más rápido que predecir tweet por tweet, sobre todo con GPU).

In [ ]:
resultados = analyzer.predict(list(test['text_prep']), batch_size=32)

test['pred_HS'] = [1 if 'hateful' in r.output else 0 for r in resultados]
test['proba_hate'] = [r.probas['hateful'] for r in resultados]
test['proba_targeted'] = [r.probas['targeted'] for r in resultados]
test['proba_aggressive'] = [r.probas['aggressive'] for r in resultados]

test[['text', 'HS', 'pred_HS', 'proba_hate']].sample(5, random_state=42)

# Evaluación

Calculamos las mismas métricas que reportamos en cap0, para poder comparar directamente.

In [ ]:
y_true = test['HS']
y_pred = test['pred_HS']
y_proba = test['proba_hate']

print(classification_report(y_true, y_pred, target_names=['No odio', 'Odio']))

metricas_pysentimiento = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred),
    'recall': recall_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred),
    'roc_auc': roc_auc_score(y_true, y_proba),
}
metricas_pysentimiento

In [ ]:
# Matriz de confusión
blue_ramp = mcolors.LinearSegmentedColormap.from_list(
    'blue_seq', ['#fcfcfb', '#cde2fb', '#86b6ef', '#3987e5', '#184f95']
)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
fig.patch.set_facecolor('#fcfcfb')
sns.heatmap(
    cm, annot=True, fmt='d', cmap=blue_ramp, cbar=False,
    linewidths=2, linecolor='#fcfcfb',
    xticklabels=['No odio', 'Odio'], yticklabels=['No odio', 'Odio'], ax=ax
)
ax.set_facecolor('#fcfcfb')
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — pysentimiento (hateful)')
plt.tight_layout()
plt.show()

## Análisis de umbral

`pred_HS` usa el umbral por defecto de 0,5 sobre `proba_hate`. Pero ese umbral es una decisión de **política de moderación**, no algo que imponga el modelo: subirlo prioriza precisión (menos falsos positivos, se banean menos tweets inocentes) y bajarlo prioriza recall (menos falsos negativos, se detecta más odio real pero con más falsas alarmas). Veamos cómo varían precisión, recall y F1 según el umbral elegido.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#fcfcfb')
ax.set_facecolor('#fcfcfb')
ax.plot(thresholds, precisions[:-1], color='#2a78d6', label='Precisión')
ax.plot(thresholds, recalls[:-1], color='#e34948', label='Recall')
ax.plot(thresholds, f1_scores[:-1], color='#898781', label='F1', linestyle='--')
ax.axvline(0.5, color='#898781', alpha=0.4, linestyle=':', label='Umbral por defecto (0.5)')
ax.axvline(best_threshold, color='#184f95', alpha=0.6, linestyle=':', label=f'Mejor F1 (umbral={best_threshold:.2f})')
ax.set_xlabel('Umbral sobre proba_hate')
ax.set_ylabel('Score')
ax.set_title('Precisión / Recall / F1 según el umbral de decisión')
ax.legend()
for spine in ax.spines.values():
    spine.set_color('#898781')
plt.tight_layout()
plt.show()

print(f'F1 con umbral 0.5: {f1_score(y_true, y_pred):.3f}')
print(f'Mejor F1 posible: {f1_scores[best_idx]:.3f}, con umbral {best_threshold:.3f}')

# Dónde se equivoca el modelo

Miramos los casos donde el modelo se equivoca con más confianza: falsos positivos (predijo odio y no había) y falsos negativos (no detectó odio que sí estaba).

## Falsos positivos

In [ ]:
falsos_positivos = test[(test['HS'] == 0) & (test['pred_HS'] == 1)].sort_values('proba_hate', ascending=False)
print(f'Cantidad de falsos positivos: {len(falsos_positivos)}')
falsos_positivos[['text', 'target', 'proba_hate']].head(8)

### Qué explican estos ejemplos

Los falsos positivos suelen concentrarse en tweets con **insultos o lenguaje agresivo que no está dirigido a un grupo protegido** (migrantes o mujeres como colectivo), sino a una persona puntual, un rival deportivo o político, o simplemente vocabulario cargado (puteadas, ironía). El modelo aprendió a asociar ciertas palabras con odio, pero HatEval etiqueta `HS=1` solo cuando el odio apunta específicamente a esos dos grupos objetivo — la definición de la tarea es más estrecha que "lenguaje ofensivo" en general.

## Falsos negativos

In [ ]:
falsos_negativos = test[(test['HS'] == 1) & (test['pred_HS'] == 0)].sort_values('proba_hate', ascending=True)
print(f'Cantidad de falsos negativos: {len(falsos_negativos)}')
falsos_negativos[['text', 'target', 'proba_hate']].head(8)

### Qué explican estos ejemplos

Los falsos negativos suelen ser casos de odio **implícito o sin insultos explícitos**: comentarios que naturalizan estereotipos, generalizaciones ("todos los inmigrantes son...") dichas en tono aparentemente neutral, sarcasmo, o jerga y abreviaciones que el modelo no asocia fuertemente con la clase odiosa. Es el mismo tipo de error que cometían los modelos de cap0 (TF-IDF y embeddings no captan ironía), aunque acá esperamos que ocurra con menos frecuencia porque el Transformer sí modela algo de contexto.

## Desempeño por target

HatEval cubre dos tipos de odio: hacia migrantes (`mig`) y hacia mujeres (`mis`), en proporciones casi iguales en el test (800 vs. 799). Veamos si el modelo funciona igual de bien en ambos.

In [ ]:
filas = []
for t in sorted(test['target'].unique()):
    sub = test[test['target'] == t]
    filas.append({
        'target': t,
        'n': len(sub),
        'precision': precision_score(sub['HS'], sub['pred_HS']),
        'recall': recall_score(sub['HS'], sub['pred_HS']),
        'f1': f1_score(sub['HS'], sub['pred_HS']),
    })
desempeno_por_target = pd.DataFrame(filas).set_index('target')
desempeno_por_target

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
fig.patch.set_facecolor('#fcfcfb')
ax.set_facecolor('#fcfcfb')
desempeno_por_target[['precision', 'recall', 'f1']].plot(
    kind='bar', ax=ax, color=['#2a78d6', '#e34948', '#898781'], rot=0
)
ax.set_title('Desempeño por target (mig vs. mis)')
ax.set_ylim(0, 1)
for spine in ax.spines.values():
    spine.set_color('#898781')
plt.tight_layout()
plt.show()

### Qué explican estos números

Si aparece una brecha entre `mig` y `mis`, vale la pena preguntarse por qué: puede deberse a que un tipo de odio se expresa de forma más explícita o estereotipada (más fácil de detectar), a un desbalance en cuántos ejemplos de cada tipo vio el modelo durante el fine-tuning sobre el train de HatEval, o a que el vocabulario asociado a un target esté más presente en el corpus de tuits (RoBERTuito) con el que se preentrenó el modelo base. Es un buen disparador de discusión sobre sesgos algorítmicos en moderación de contenido: un modelo puede funcionar de forma dispar según el grupo afectado, incluso con buen desempeño agregado.

# Comparación con las representaciones clásicas (cap0)

La notebook de [Vectorización](<../cap0/00 - SICSS-BAires - Vectorización.ipynb>) evaluó, sobre este mismo test, tres formas de representar el texto (TF, TF-IDF, embeddings promediados) con una regresión logística LASSO. Completá la fila de `pysentimiento` con los números que obtuviste arriba y compará:

| Modelo | Accuracy | Precisión | Recall | F1 | ROC AUC |
|---|---|---|---|---|---|
| TF + LASSO | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* |
| TF-IDF + LASSO | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* |
| Embeddings + LASSO | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* | *(ver cap0)* |
| **pysentimiento** | ejecutar arriba | ejecutar arriba | ejecutar arriba | ejecutar arriba | ejecutar arriba |

*Los valores de las primeras tres filas están en la celda `# Conclusiones` de `cap0/00 - SICSS-BAires - Vectorización.ipynb`; la fila de pysentimiento se completa con el diccionario `metricas_pysentimiento` calculado más arriba, corriendo esta notebook.*

In [ ]:
metricas_pysentimiento

# Conclusiones

- `pysentimiento` nos da, con **una sola línea** (`create_analyzer` + `.predict()`), un modelo que en cap0 nos llevó varias celdas de ingeniería de features y búsqueda de hiperparámetros construir. Esa es la principal ventaja práctica de un modelo preentrenado: no hay que diseñar features ni entrenar nada.
- La representación contextual de un Transformer (RoBERTuito) capta relaciones entre palabras y algo de orden/sintaxis que TF y TF-IDF, por construcción, no pueden ver — cada tweet es más que una bolsa de palabras.
- Esto tiene costos: se necesita descargar un modelo de cientos de MB, correrlo es más lento que una regresión logística (especialmente sin GPU), y **dependemos de las decisiones de quien entrenó el modelo** (qué datos usó, qué definición de "odio" aplicó) en vez de controlar nosotros todo el pipeline.
- Recordá el caveat del principio: estos números son sobre el test *held-out* de HatEval, el mismo corpus con el que se entrenó el modelo. No los uses para afirmar qué tan bien funcionaría este mismo analyzer sobre otro dominio (por ejemplo, los tweets de campaña electoral en `data/tweets_*.zip`) sin evaluarlo ahí primero.

# Ejercicio

1. **Etiquetas multilabel.** Hasta acá solo evaluamos `hateful` vs. `HS`. Repetí el análisis de la sección `# Evaluación` para `targeted` vs. `TR` y `aggressive` vs. `AG`, restringiéndote a las filas donde `HS == 1` (que es donde `TR`/`AG` están definidas). ¿El modelo distingue mejor si el odio es agresivo o si está dirigido a alguien en particular?
2. **Sentimiento.** Creá un segundo analyzer con `create_analyzer(task="sentiment", lang="es")` y corrélo sobre los mismos tweets. ¿Los tweets con `HS=1` tienden a tener sentimiento negativo? ¿Hay tweets con sentimiento negativo que *no* son discurso de odio? ¿Qué te dice eso sobre la diferencia entre "sentimiento negativo" y "discurso de odio" como conceptos?

In [ ]:
###